# Importing libraries

In [1]:
# Basic libraries
import time
import torch
import pandas as pd
from datasets import load_dataset

# Transformers library for LLM
from transformers import AutoTokenizer, AutoModelForCausalLM

# Utilities and metrics
from memory_profiler import memory_usage
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Instancing LLM

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
model_name_or_path = 'meta-llama/Llama-3.2-3B'

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype=torch.float16)

model.to(device)

print(next(model.parameters()).dtype)

Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.97s/it]


torch.float16


# Defining a function for classifying a given text

In [26]:
def classify_text(text, labels):
    # Reset peak memory stats before starting
    torch.cuda.reset_peak_memory_stats(device)
    start_memory = torch.cuda.memory_allocated(device)

    labels_str = ', '.join(labels)
    prompt = (
        f"Classify the following text into one of the categories: {labels_str}.\n\n"
        f"Text: {text}\n\n"
        "Only output the category name without any additional text.\n\n"
        "Category:"
    )

    print(f"Prompt:\n{prompt}\n")
    print("-"*100)
    
    # Tokenize the input and move to GPU
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    # Calculate the maximum label length in tokens
    max_label_length = max([len(tokenizer.encode(label, add_special_tokens=False)) for label in labels])
    print(f"Max label length: {max_label_length}")
    print("-"*100)
    
    # Generate the model's output with limited tokens
    start = time.perf_counter()
    output = model.generate(
        **inputs,
        max_new_tokens=max_label_length,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        temperature=None,
        top_p=None,
        do_sample=False, # Greedy decoding, always choose the token with the highest probability
    )
    total_time = time.perf_counter() - start
    
    # Decode only the newly generated tokens
    generated_tokens = output[0][inputs['input_ids'].shape[-1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    print(f"Generated text:\n{generated_text}")
    print("-"*100)
    
    # Extract the category from the generated text
    category = generated_text.strip().split('\n')[0]
    print(f"Category: {category}")
    print("-"*100)

    # Get peak memory usage
    max_memory = torch.cuda.max_memory_allocated(device)
    peak_vram = (max_memory - start_memory) / 1024 ** 2  # Convert to MB
    print(f"Peak GPU memory usage for this function: {peak_vram:.2f} MB")
    print("-"*100)
    
    return category, total_time, peak_vram


# Importing dataset

In [27]:
ds = load_dataset("cardiffnlp/tweet_eval", "hate")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2970 entries, 0 to 2969
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2970 non-null   object
 1   label   2970 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 46.5+ KB


In [28]:
# Example test data
test_texts = [
    'The stock market crashed today.',
    'The team won the championship!',
    'New technology advances in AI.',
    'Delicious recipes for homemade pasta.',
    'The onset of LLMs has revolutionized the field of computer science.'
]
true_labels = ['Business', 'Sports', 'Technology', 'Cooking', 'Technology']

# List of possible labels
labels = ['Business', 'Sports', 'Technology', 'Cooking']

# Classification

In [29]:
columns = ['text', 'true_label', 'predicted_label', 'inference_time', 'peak_vram_usage']

results = pd.DataFrame(columns=columns)

In [30]:
predicted_labels = []

for text in test_texts:
    pred, total_time, peak_vram = classify_text(text, labels)
    
    result = {
        'text': text,
        'true_label': true_labels[test_texts.index(text)],
        'predicted_label': pred,
        'inference_time': total_time,
        'peak_vram_usage': peak_vram,
    }

    results = pd.concat([results, pd.DataFrame([result], columns=columns)], ignore_index=True)

Prompt:
Classify the following text into one of the categories: Business, Sports, Technology, Cooking.

Text: The stock market crashed today.

Only output the category name without any additional text.

Category:

----------------------------------------------------------------------------------------------------
Max label length: 2
----------------------------------------------------------------------------------------------------
Generated text:
 Business


----------------------------------------------------------------------------------------------------
Category: Business
----------------------------------------------------------------------------------------------------
Peak GPU memory usage for this function: 7.21 MB
----------------------------------------------------------------------------------------------------
Prompt:
Classify the following text into one of the categories: Business, Sports, Technology, Cooking.

Text: The team won the championship!

Only output the categor

C:\Users\Rafael\AppData\Local\Temp\ipykernel_23440\3394573034.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([result], columns=columns)], ignore_index=True)


Generated text:
 Cooking


----------------------------------------------------------------------------------------------------
Category: Cooking
----------------------------------------------------------------------------------------------------
Peak GPU memory usage for this function: 7.21 MB
----------------------------------------------------------------------------------------------------
Prompt:
Classify the following text into one of the categories: Business, Sports, Technology, Cooking.

Text: The onset of LLMs has revolutionized the field of computer science.

Only output the category name without any additional text.

Category:

----------------------------------------------------------------------------------------------------
Max label length: 2
----------------------------------------------------------------------------------------------------
Generated text:
 Technology


----------------------------------------------------------------------------------------------------


In [31]:
results

,text,true_label,predicted_label,inference_time,peak_vram_usage
0,The stock market crashed today.,Business,Business,0.407878,7.211426
1,The team won the championship!,Sports,Sports,0.100083,7.211426
2,New technology advances in AI.,Technology,Technology,0.097807,7.211426
3,Delicious recipes for homemade pasta.,Cooking,Cooking,0.096202,7.211426
4,The onset of LLMs has revolutionized the field...,Technology,Technology,0.095617,8.833496


# Analysing results

In [32]:
accuracy = accuracy_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels, average='weighted')
recall = recall_score(true_labels, predicted_labels, average='weighted')
precision = precision_score(true_labels, predicted_labels, average='weighted')

print(f'Accuracy: {accuracy:.2f}')
print(f'F1 Score: {f1:.2f}')
print(f'Recall: {recall:.2f}')
print(f'Precision: {precision:.2f}')

print(f'True Labels: {true_labels}')
print(f'Predicted Labels: {predicted_labels}')


ValueError: Found input variables with inconsistent numbers of samples: [5, 0]